In [ ]:
import csv
import os
import sys
import pandas as pd
import numpy as np
from emmaemb.core import Emma
sys.path.append('/home/unix/vkrhk/EmmaEmb/analysis')

IMG_OUTPUT_PATH = '/home/unix/vkrhk/EmmaEmb/img'
DATA_PATH = '/home/unix/vkrhk/EmmaEmb/data'
EMBEDDINGS_PATH = '/media/drive2/vkrhk/embeddings'
SUBSET_SIZE = 50 # take first N proteins from each dataset for testing

emb_spaces = [("ESM2", f"{EMBEDDINGS_PATH}/esm2_t33_650M_UR50D/layer_33/chopped_1022_overlap_300"),
("ANKH", f"{EMBEDDINGS_PATH}/ankh_base/layer_None/chopped_1022_overlap_300"),
("ProstT5", f"{EMBEDDINGS_PATH}/Rostlab/ProstT5/layer_None/chopped_1022_overlap_300/"),
("ProtT5", f"{EMBEDDINGS_PATH}/Rostlab/prot_t5_xl_uniref50/layer_None/chopped_1022_overlap_300"),
("ESM1", f"{EMBEDDINGS_PATH}/esm1_t34_670M_UR100/layer_34/chopped_1022_overlap_300/"),
("ESMC", f"{EMBEDDINGS_PATH}/esmc-300m-2024-12/layer_None/chopped_1022_overlap_300/"),]

# colect data:
feature_data = []
embeddings = {}

for dataset in ['train.txt', 'scPDB_enhanced_binding_sites_translated.csv']:
    with open(f'{DATA_PATH}/{dataset}', 'r') as f:
        reader = csv.reader(f, delimiter=';')
        for ii, row in enumerate(reader):
            protein_id = row[0] + row[1]
            annotation = row[3].split(' ')
            annotation = [int(i[1:]) for i in annotation]
            sequence = row[4]

            these_embeddings = {}
            length, have_same_length = 0, True
            for i, (embeddings_name, embeddings_path) in enumerate(emb_spaces):
                path = f"{embeddings_path}/{protein_id}.npy"
                if not os.path.exists(path):
                    have_same_length = False
                    break
                embedding = np.load(path)
                these_embeddings[embeddings_name] = embedding
                # check that all embedding spaces have the same length for this protein
                if i == 0:
                    length = embedding.shape[0]
                else:
                    if embedding.shape[0] != length:
                        have_same_length = False
                        break

            if not have_same_length:
                continue
            
            BINDING_FLAG = 'CRYPTIC-BINDING' if dataset == 'train.txt' else 'BINDING'
            for i in range(len(sequence)):
                feature_data.append([sequence[i], BINDING_FLAG if i in annotation else 'NON-BINDING'])
            
            for embeddings_name in these_embeddings:
                if embeddings_name not in embeddings:
                    embeddings[embeddings_name] = []
                embeddings[embeddings_name].append(these_embeddings[embeddings_name])
            if ii > SUBSET_SIZE:
                break
    
for embeddings_name in embeddings:
    concatenated_embeddings_path = f"{DATA_PATH}/concatenated-embeddings/{embeddings_name}_binding_site_embeddings.npy"
    embeddings[embeddings_name] = np.concatenate(embeddings[embeddings_name], axis=0)
    np.save(concatenated_embeddings_path, embeddings[embeddings_name])  

feature_data = pd.DataFrame.from_records(feature_data, columns=["amino acid", "binding_site"])

# initiate Emma object and load embedding spaces
emma = Emma(feature_data=feature_data)
for embeddings_name, _ in emb_spaces:
    emma.add_emb_space(
        embeddings_source=f"{DATA_PATH}/concatenated-embeddings/{embeddings_name}_binding_site_embeddings.npy",
        emb_space_name=embeddings_name)


33389 samples loaded.
Categories in meta data: ['binding_site']
Numerical columns in meta data: []
Embedding space 'ESM2' added successfully.
Embeddings have 1280 features each.
Embedding space 'ANKH' added successfully.
Embeddings have 1536 features each.
Embedding space 'ProstT5' added successfully.
Embeddings have 1024 features each.
Embedding space 'ProtT5' added successfully.
Embeddings have 1024 features each.
Embedding space 'ESM1' added successfully.
Embeddings have 1280 features each.
Embedding space 'ESMC' added successfully.
Embeddings have 960 features each.


In [ ]:
from emmaemb.vizualisation import plot_knn_alignment_across_classes

for embeddings_name, _ in emb_spaces:
    emma.calculate_pairwise_distances(emb_space=embeddings_name, metric='euclidean')

fig_3 = plot_knn_alignment_across_classes(
    emma, k=3, metric='euclidean', feature='binding_site')
fig_3.show()



Calculating pairwise distances using euclidean...
Using GPU for distance calculation.


Computing pairwise distances (euclidean): 100%|███████████████████████████████████████████████████████████████████████████████████| 334/334 [00:02<00:00, 118.52it/s]


Calculating pairwise distances using euclidean...
Using GPU for distance calculation.


Computing pairwise distances (euclidean): 100%|███████████████████████████████████████████████████████████████████████████████████| 334/334 [00:02<00:00, 118.31it/s]


Calculating pairwise distances using euclidean...
Using GPU for distance calculation.


...

Computing pairwise distances (euclidean): 100%|███████████████████████████████████████████████████████████████████████████████████| 334/334 [00:02<00:00, 149.11it/s]


Calculating pairwise distances using euclidean...
Using GPU for distance calculation.


Computing pairwise distances (euclidean): 100%|███████████████████████████████████████████████████████████████████████████████████| 334/334 [00:02<00:00, 120.29it/s]


Calculating pairwise distances using euclidean...
Using GPU for distance calculation.


Computing pairwise distances (euclidean): 100%|███████████████████████████████████████████████████████████████████████████████████| 334/334 [00:02<00:00, 129.53it/s]


In [24]:
fig_3 = plot_knn_alignment_across_classes(
    emma, k=5, metric='euclidean', feature='binding_site')
fig_3.show()


In [20]:
feature_data[feature_data['binding_site'] == 'CRYPTIC-BINDING'] = 'BINDING'
emma2 = Emma(feature_data=feature_data)

for embeddings_name, _ in emb_spaces:
    emma2.add_emb_space(
        embeddings_source=f"{DATA_PATH}/concatenated-embeddings/{embeddings_name}_binding_site_embeddings.npy",
        emb_space_name=embeddings_name)

33389 samples loaded.
Categories in meta data: ['binding_site']
Numerical columns in meta data: []
Embedding space 'ESM2' added successfully.
Embeddings have 1280 features each.
Embedding space 'ANKH' added successfully.
Embeddings have 1536 features each.
Embedding space 'ProstT5' added successfully.
Embeddings have 1024 features each.
Embedding space 'ProtT5' added successfully.
Embeddings have 1024 features each.
Embedding space 'ESM1' added successfully.
Embeddings have 1280 features each.
Embedding space 'ESMC' added successfully.
Embeddings have 960 features each.


In [22]:
for embeddings_name, _ in emb_spaces:
    emma2.calculate_pairwise_distances(emb_space=embeddings_name, metric='euclidean')

fig_3 = plot_knn_alignment_across_classes(
    emma2, k=10, metric='euclidean', feature='binding_site')
fig_3.show()


Calculating pairwise distances using euclidean...
Using GPU for distance calculation.


Computing pairwise distances (euclidean): 100%|███████████████████████████████████████████████████████████████████████████████████| 334/334 [00:02<00:00, 126.17it/s]


Calculating pairwise distances using euclidean...
Using GPU for distance calculation.


Computing pairwise distances (euclidean): 100%|███████████████████████████████████████████████████████████████████████████████████| 334/334 [00:01<00:00, 169.16it/s]


Calculating pairwise distances using euclidean...
Using GPU for distance calculation.


...

Computing pairwise distances (euclidean): 100%|███████████████████████████████████████████████████████████████████████████████████| 334/334 [00:01<00:00, 194.05it/s]


Calculating pairwise distances using euclidean...
Using GPU for distance calculation.


Computing pairwise distances (euclidean): 100%|███████████████████████████████████████████████████████████████████████████████████| 334/334 [00:01<00:00, 171.88it/s]


Calculating pairwise distances using euclidean...
Using GPU for distance calculation.


Computing pairwise distances (euclidean): 100%|███████████████████████████████████████████████████████████████████████████████████| 334/334 [00:02<00:00, 166.04it/s]
